<a href="https://colab.research.google.com/github/Hassan-Himaz/GM1-Crutch/blob/main/step%20detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def detect_crutch_steps(
    csv_path,
    body_mass_kg=70,
    force_col="force",
    time_col="time",
    threshold_fraction=0.05,
    min_duration=0.15,
    max_duration=3.0,
    smoothing_window=5,
):
    df = pd.read_csv(csv_path)

    for col in [time_col, force_col]:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    df = df.sort_values(time_col).reset_index(drop=True)

    df["force_smooth"] = (
        df[force_col]
        .rolling(window=smoothing_window, center=True, min_periods=1)
        .mean()
    )

    if {"ax", "ay", "az"}.issubset(df.columns):
        df["acc_mag"] = np.sqrt(df["ax"]**2 + df["ay"]**2 + df["az"]**2)
    else:
        df["acc_mag"] = np.nan

    if {"gx", "gy", "gz"}.issubset(df.columns):
        df["gyro_mag"] = np.sqrt(df["gx"]**2 + df["gy"]**2 + df["gz"]**2)
    else:
        df["gyro_mag"] = np.nan

    threshold_n = threshold_fraction * body_mass_kg * 9.81
    above = df["force_smooth"] > threshold_n

    events = []
    in_event = False
    start_idx = None

    for i, is_above in enumerate(above):
        if is_above and not in_event:
            start_idx = i
            in_event = True

        elif not is_above and in_event:
            end_idx = i - 1
            in_event = False

            event_df = df.iloc[start_idx:end_idx + 1]
            start_time = event_df[time_col].iloc[0]
            end_time = event_df[time_col].iloc[-1]
            duration = end_time - start_time

            if duration < min_duration or duration > max_duration:
                continue

            peak_force = event_df["force_smooth"].max()
            peak_idx = event_df["force_smooth"].idxmax()
            peak_time = df.loc[peak_idx, time_col]

            impulse = np.trapezoid(
                event_df["force_smooth"],
                event_df[time_col]
            )

            events.append({
                "start_idx": start_idx,
                "end_idx": end_idx,
                "start_time": start_time,
                "end_time": end_time,
                "peak_time": peak_time,
                "duration_s": duration,
                "peak_force_n": peak_force,
                "impulse_ns": impulse,
                "peak_acc": event_df["acc_mag"].max(),
                "peak_gyro": event_df["gyro_mag"].max(),
            })

    if in_event:
        end_idx = len(df) - 1
        event_df = df.iloc[start_idx:end_idx + 1]
        start_time = event_df[time_col].iloc[0]
        end_time = event_df[time_col].iloc[-1]
        duration = end_time - start_time

        if min_duration <= duration <= max_duration:
            peak_force = event_df["force_smooth"].max()
            peak_idx = event_df["force_smooth"].idxmax()
            peak_time = df.loc[peak_idx, time_col]

            impulse = np.trapezoid(
                event_df["force_smooth"],
                event_df[time_col]
            )

            events.append({
                "start_idx": start_idx,
                "end_idx": end_idx,
                "start_time": start_time,
                "end_time": end_time,
                "peak_time": peak_time,
                "duration_s": duration,
                "peak_force_n": peak_force,
                "impulse_ns": impulse,
                "peak_acc": event_df["acc_mag"].max(),
                "peak_gyro": event_df["gyro_mag"].max(),
            })

    events_df = pd.DataFrame(events)
    return df, events_df, threshold_n


def plot_detected_steps(df, events_df, threshold_n, time_col="time"):
    plt.figure(figsize=(12, 5))
    plt.plot(df[time_col], df["force_smooth"], label="Smoothed force")
    plt.axhline(threshold_n, linestyle="--", label="5% body-weight threshold")

    for _, event in events_df.iterrows():
        plt.axvspan(event["start_time"], event["end_time"], alpha=0.2)

    plt.xlabel("Time (s)")
    plt.ylabel("Force (N)")
    plt.title("Detected crutch step events")
    plt.legend()
    plt.show()


# For Google Colab: upload your CSV first
from google.colab import files
uploaded = files.upload()

csv_filename = list(uploaded.keys())[0]

df, steps, threshold = detect_crutch_steps(
    csv_filename,
    body_mass_kg=70
)

print(f"Threshold used: {threshold:.2f} N")
print("\nDetected steps:")
print(steps)

plot_detected_steps(df, steps, threshold)

KeyboardInterrupt: 